# Jour 1 · Retrouver un document sans utiliser de LLM


## Pourquoi commencer sans LLM ?

Le modèle ne peut utiliser que les passages transmis. Si la recherche retourne une procédure d'imprimante pour une question VPN, le meilleur prompt ne recréera pas la procédure absente.

```text
question → embedding → Chroma → documents
```


## L'objet `Document`

Un texte répond à « que dit le passage ? ». Les métadonnées répondent à « d'où vient-il ? ». Les deux restent attachés dans un objet `Document`.


![Anatomie d'un objet Document](../../assets/jour_01/01_anatomie_document.png)

*`page_content` porte le savoir ; `metadata` conserve les informations nécessaires pour filtrer et citer.*


In [ ]:
from pathlib import Path


def find_course_root() -> Path:
    """Retrouve la racine, même si Jupyter démarre dans un sous-dossier."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "documents").is_dir() and (candidate / "jour_01").is_dir():
            return candidate
    raise FileNotFoundError("Racine du cours introuvable.")


ROOT = find_course_root()
DOCUMENTS_DIR = ROOT / "documents"
OUTPUTS_DIR = ROOT / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

print("Racine du cours :", ROOT)


In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

CHAT_MODEL = "qwen2.5:3b"
EMBEDDING_MODEL = "bge-m3"

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
llm = ChatOllama(model=CHAT_MODEL, temperature=0)

print("Modèle de génération :", CHAT_MODEL)
print("Modèle d'embeddings  :", EMBEDDING_MODEL)


In [ ]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content=(
            "Pour un écran noir, vérifier le voyant du moniteur, le câble d'alimentation, "
            "le câble HDMI ou DisplayPort et la source vidéo sélectionnée."
        ),
        metadata={"source": "procedure_ecran.md", "categorie": "materiel"},
    ),
    Document(
        page_content=(
            "Pour se connecter à distance, ouvrir GlobalProtect, saisir le portail "
            "vpn.entreprise.test puis valider le code MFA."
        ),
        metadata={"source": "procedure_vpn.md", "categorie": "reseau"},
    ),
    Document(
        page_content=(
            "En cas de bourrage, éteindre l'imprimante, retirer doucement le papier "
            "dans le sens de circulation puis contrôler les rouleaux."
        ),
        metadata={"source": "procedure_imprimante.md", "categorie": "materiel"},
    ),
    Document(
        page_content=(
            "Un technicien ne demande jamais un mot de passe ni un code MFA. "
            "Ces secrets ne doivent pas être placés dans un ticket."
        ),
        metadata={"source": "politique_securite.md", "categorie": "securite"},
    ),
]

print(f"{len(documents)} documents préparés")


In [ ]:
premier_document = documents[0]
print("CONTENU :", premier_document.page_content)
print("MÉTADONNÉES :", premier_document.metadata)


## De la phrase au vecteur

Un **embedding** transforme un texte en liste de nombres. Chaque nombre isolé n'a pas une étiquette lisible comme « VPN ». C'est la position globale du vecteur qui devient utile.

Deux textes proches par le sens produisent généralement des vecteurs proches. Le schéma réduit volontairement les nombreuses dimensions réelles à deux axes visibles.


### Un vecteur, sans faire de mathématiques

Un nombre permet de placer un point sur une ligne. Plusieurs nombres permettent de placer un point dans un espace comportant plusieurs directions. Cette liste de nombres s'appelle ici un **vecteur**.

Le modèle d'embeddings calcule ce vecteur pour nous. Nous n'interprétons pas chaque nombre séparément : nous observons surtout si deux textes obtiennent des positions proches, donc s'ils parlent probablement de choses proches.


![Espace simplifié regroupant des phrases par sujet](../../assets/jour_01/01_espace_embeddings.png)

*Les coordonnées sont pédagogiques ; les vrais embeddings de `bge-m3` possèdent beaucoup plus de dimensions.*


In [ ]:
vector = embeddings.embed_query("Je n'ai plus d'image sur mon moniteur")
print("Type :", type(vector))
print("Nombre de dimensions :", len(vector))
print("Cinq premières valeurs :", [round(value, 4) for value in vector[:5]])


## Stocker pour rechercher

Chroma conserve le contenu, les métadonnées et l'embedding. Nous utilisons d'abord un client **éphémère**, présent seulement pendant l'exécution du notebook. La persistance viendra lorsque nous saurons éviter les doublons.


In [ ]:
import chromadb
from langchain_chroma import Chroma

client = chromadb.EphemeralClient()
vectorstore = Chroma(
    client=client,
    collection_name="helpdesk_demo",
    embedding_function=embeddings,
)
vectorstore.add_documents(
    documents,
    ids=[f"doc-{index}" for index in range(len(documents))],
)

print("Documents indexés :", len(vectorstore.get()["ids"]))


## Faire une première recherche

`similarity_search_with_score` renvoie des couples `(document, distance)`. Ici, une distance plus petite signifie que les vecteurs sont plus proches.

Une autre API peut renvoyer une similarité où une grande valeur est meilleure. Il faut donc connaître le sens du score utilisé.


### Lire le résultat sans se faire piéger par le score

La recherche classe d'abord les passages du plus proche au plus éloigné. Dans cette méthode, une petite **distance** est meilleure qu'une grande distance.

Le nombre exact n'est pas une note universelle sur 10 : il dépend du modèle, de la base et de la mesure utilisée. Pour commencer, vérifiez surtout l'ordre des documents et leur contenu avant de choisir un éventuel seuil.


In [ ]:
question = "Je travaille de chez moi. Quelle adresse saisir dans GlobalProtect ?"
resultats = vectorstore.similarity_search_with_score(question, k=3)

for rang, (document, distance) in enumerate(resultats, start=1):
    print(f"{rang}. distance={distance:.3f} · {document.metadata['source']}")
    print("  ", document.page_content)


## Comprendre top-k

`k=3` demande trois candidats. Cela ne signifie pas que les trois sont corrects : Chroma retourne les meilleurs éléments disponibles, même pour une question sans rapport avec la base.


![Sélection des voisins les plus proches](../../assets/jour_01/01_top_k.png)

*Les points verts sont les meilleurs candidats ; leur pertinence doit encore être évaluée.*


## Du vector store au retriever

Un **retriever** reçoit une question et renvoie une `list[Document]`. Cette interface simple pourra être insérée dans une chaîne LCEL.


In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
documents_retrouves = retriever.invoke(question)

print(type(documents_retrouves))
for document in documents_retrouves:
    print("-", document.metadata["source"])


### À vous de jouer — tester une reformulation

Posez deux questions différentes décrivant un écran noir. Vérifiez si la même source arrive en première position malgré le changement de vocabulaire.


In [ ]:
# Correction — comparer deux formulations du même besoin
questions_exercice = [
    "Je n'ai aucune image sur mon moniteur.",
    "Mon écran reste noir.",
]
for question_exercice in questions_exercice:
    premier = vectorstore.similarity_search(question_exercice, k=1)[0]
    print(question_exercice, "→", premier.metadata["source"])


## Un test hors périmètre

La base ne contient rien sur la météo. Le retriever retourne pourtant les documents les plus proches disponibles. Le prompt devra donc autoriser explicitement le refus de répondre.


In [ ]:
hors_perimetre = vectorstore.similarity_search_with_score(
    "Quel temps fera-t-il demain à Bruxelles ?",
    k=2,
)
for document, distance in hors_perimetre:
    print(f"distance={distance:.3f} · {document.metadata['source']}")


## À retenir

- Un `Document` associe contenu et métadonnées.
- Un embedding représente le sens sous forme de vecteur.
- Chroma recherche des vecteurs proches.
- Un retriever renvoie une liste de documents.
- top-k signifie « meilleurs candidats », pas « réponses garanties ».
